# பாடம் 09 - மேட்டா-அறிவு வடிவமைப்பு முறை


## அமைப்பு

இந்த நோட்புக் Microsoft Agent Framework-ஐப் பயன்படுத்தி Metacognition வடிவமைப்பு மாதிரியை விளக்குகிறது.

**முன் தேவைகள்:**
- சூழல் மாறிலிகள் மூலம் கட்டமைக்கப்பட்ட Azure OpenAI deployment
- Azure CLI அங்கீகரிக்கப்பட்டது (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## மெட்டகோக்னிஷன் என்றால் என்ன?

மெட்டகோக்னிஷன் என்பது **சிந்தனை பற்றி சிந்தித்தல்**. ஏ.ஐ. ஏஜென்ட்களின் சூழலில், இதன் பொருள் பின்வருவனச் செய்யக்கூடிய ஏஜென்ட்களை உருவாக்குவதாகும்:

- **சுய-பரிசீலனை செய்ய** அவர்கள் சொந்த வெளியீடுகள் மற்றும் யோசனை செயல்முறை குறித்து
- **பிழைகளை கண்டறிதல்** மற்றும் மௌனமாக தோல்வியடையாமல் சீராக மீட்குதல்
- **மதிப்பீடு செய்ய** அவர்கள் பதில்கள் முழுமையாகவும் உதவியாகவும் உள்ளதா என்பதை
- **திட்டத்தை மாற்ற** அவர்கள் ஆரம்ப அணுகுமுறை வேலை செய்யாதபோது (உதா., காப்பு அமைப்பிற்கு திரும்புதல்)

ஒரு மெட்டகோக்னிஷன் ஏஜென்ட் கேள்விகளுக்கு மட்டும் பதில் சொல்லுவதில்லை — அது தனது சொந்த செயல்திறனைக் கண்காணித்து உடனுக்குடன் அவற்றைத் திருத்திக் கொள்கிறது.


## முதன்மை மற்றும் காப்பு கருவிகள்

ஒரு பொதுவான மேட்டகோக்னிஷன் முறை என்பது **பின்வாங்கும் திட்டம்**. ஏஜென்ட் முதலில் ஒரு முதன்மை கருவியை முயற்சிக்கிறது; அது தோல்வியடைந்தால் (உதாரணமாக, 404 பிழை), ஏஜென்ட் அந்த தோல்வியை உணர்ந்து வெளிப்படையாக காப்பு கருவிக்கு மாறுகிறது.

இது உண்மையான உலகில் முதன்மை சேவைகள் கிடைக்காமல் இருக்கக்கூடிய சூழ்நிலைகளை பிரதிபலிக்கிறது, மற்றும் ஏஜென்டுகள் மாற்று பாதையைத் தேர்வு செய்வதற்கு முன் தாமாகவே பிரச்சனையை கண்டறிய வேண்டும்.

கீழே நாங்கள் இரண்டு விமான தேடல் கருவிகளை வரையறுக்கிறோம்:
- **முதன்மை** — பாரிஸ், டோக்கியோ மற்றும் பார்சிலோனாவை உள்ளடக்கியது
- **காப்பு** — பெர்லின், சிட்னி மற்றும் நியூயார்க் சிட்டியை உள்ளடக்கியது


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## பிழை மீட்புடன் கூடிய சுய-பரிசீலனை முகவர்

கீழே உள்ள முகவருக்கு முதலில் முதன்மை விமான அமைப்பை முயற்சிக்க, தோல்விகளை அடையாளம் காண, மற்றும் வெளிப்படையாக காப்பு அமைப்பிற்கு திரும்பவேண்டும் என்று ஆதேசம் வழங்கப்பட்டுள்ளது. ஒவ்வொரு பதிலுக்குப் பிறகும் அது சுருக்கமாக தன்னைத்தானே மதிப்பீடு செய்து, அது பயனரின் கேள்விக்கு முழுமையாக பதிலளித்ததா என்பதைப் பார்க்கும்.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## தன்ன்மதிப்பீட்டு முறை

மெட்டாகாக்னிஷனின் இன்னொரு அம்சம் **தன்ன்மதிப்பீடு**: ஒரு தனி ஏஜெண்ட் (அல்லது அதே ஏஜெண்ட் இரண்டாவது முறையில்) ஒரு பதிலை முழுமை, துல்லியம் மற்றும் பயன்தன்மைக்காக மதிப்பாய்வு செய்கிறது.

கீழே நாம் `ResponseEvaluator` என்ற ஏஜெண்டை உருவாக்குகிறோம், இது பயண-ஏஜெண்ட் பதில்களை மூன்று பரிமாணங்களில் மதிப்பெண்ணிடும்.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework பயன்படுத்தி **மேட்டாகாக்னிட்டிவ் முகவரிகள்** எப்படி உருவாக்குவது என்பதை கற்றுக்கொண்டீர்கள்:

- **சுய-பரிசீலனை**: தங்கள் சொந்த காரணவியல் செயல்பாடுகளை கண்காணித்து என்ன நடந்தது என்பதை வெளிப்படையாக தொடர்பு கொள்ளும் முகவர்கள்.
- **மறுபடி முயற்சிகளுடன் பிழை மீட்பு**: முதன்மை + காப்பு கருவி மாதிரியில், முகவர் தோல்விகளை (உதாரணம்: 404 பிழைகள்) கண்டறிந்து தானாகவே மாற்று ஆதாரத்தை முயற்சிக்கிறான்.
- **சுயமதிப்பீடு**: பதில்களின் முழுமை, துல்லியம் மற்றும் பயன்தன்மைக்கு மதிப்பெண்கள் வழங்கும் தனித்தனியான மதிப்பீட்டாளர் முகவர்.

இந்த மாதிரிகள் முகவரிகளை மேலும் வலிமையான, வெளிப்படையான மற்றும் நம்பகமானவையாக ஆக்குகின்றன — உற்பத்தி இயக்கங்களில் அவை முக்கியமான பண்புகள்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
மறுப்பு அறிவிப்பு:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவையான [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சித்தாலும், தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறான பொருள் விளக்கங்கள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனியுங்கள். அதன் தாய்மொழியில் உள்ள மூல ஆவணத்தை அதிகாரபூர்வ ஆதாரமாக கருத வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதனால் ஏற்படும் ஏதேனும் தவறான புரிதல்கள் அல்லது தவறான விளக்கங்களுக்கு நாங்கள் பொறுப்பேற்கமாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
